# POMDP agent training — S_v2 real-cuBLAS CUDA backend

Notebook in the style of the upstream `experiments/pomdp_agent_training.ipynb`, but with an additional explicit `traincuda(...)` path for the S_v2 backend.

The original `ag.train(...)` method is not overwritten. CPU and CuPy still use the official olfactory-navigation training path; CUDA uses `ag_cuda.traincuda(...)`.


## 1. Imports and package path

Set `PACKAGE_ROOT` to the extracted S_v2 package. Keep this cell near the top, before importing `olfnav_cuda_*`, so the notebook uses this local package and not an older module from another directory.


In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

PACKAGE_ROOT = Path("/home/jlpfritas/HPC-POMDP/v3/package_S_v2_pomdp_agent_training_style_allstates_fix3")
PACKAGE_PY = PACKAGE_ROOT / "python"

if str(PACKAGE_PY) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PY))

# Avoid accidentally reusing stale modules from a previous package in the same kernel.
for name in list(sys.modules):
    if name.startswith("olfnav_cuda_backend") or name.startswith("olfnav_cuda_notebook"):
        del sys.modules[name]

from olfactory_navigation import Environment
from olfactory_navigation.agents import FSVI_Agent
from olfnav_cuda_notebook import (
    enable_cuda_backend,
    install_simulation_history_patch,
    normalize_environment_for_exact_converter,
    load_environment_from_metadata,
    read_environment_metadata,
    environment_kwargs_from_metadata,
    show_cuda_training_report,
    raw_start_points_from_environment,
    clean_start_points,
    generate_policy_start_points,
    run_policy_evaluation,
    run_policy_full_evaluation,
)

install_simulation_history_patch(verbose=True)

CUDA_LIB = PACKAGE_ROOT / "build" / "libpomdp_backup_cuda.so"
print("Using PACKAGE_ROOT:", PACKAGE_ROOT)
print("CUDA_LIB:", CUDA_LIB)
print("CUDA_LIB exists:", CUDA_LIB.exists())


## 2. Global configuration

Defaults are conservative. Increase `EXPANSIONS` only after a small run passes.

This notebook follows upstream `pomdp_agent_training.ipynb`: no `partitions` parameter is used. The default exact converter keeps the environment grid as the POMDP state space.

For A100, rebuild first with: `bash scripts/31_build_backend_lib.sh --arch 80 --clean`.


In [ ]:
# Environment generated from the reconstructed olfactory dataset.
ENV_DIR = Path("/home/jlpfritas/HPC-POMDP/v3/recon/Env-olfnav-trim552-fliplr-cropx780-thr1e4/Env-320_485-marg_20_20_20_20-edge_wrap_vertical-start_odor_present-source_163_406_radius2")

# Training parameters.
# No PARTITIONS here: `FSVI_Agent(env)` uses all environment states, like the upstream notebook.
SEED = 123
GAMMA = 0.95
EXPANSIONS = 100
MAX_BELIEF_GROWTH = 10
PRUNE_INTERVAL = 10
PRUNE_LEVEL = 1

# CUDA backend. Use "auto_real" for S_v2 dispatch, or force "v7" / "v8".
CUDA_DEVICE = 0
CUDA_VERSION = "auto_real"

# Toggle runs.
RUN_CUDA_TRAIN = True
RUN_CPU_TRAIN = False
RUN_CUPY_TRAIN = False
RUN_POLICY_EVAL = True
# Quick evaluation subset. Set None to use all starts in the quick-eval cell.
N_EVAL = 100
# Full all-clean-starts evaluation is intentionally off by default because it can be slow.
RUN_FULL_POLICY_EVAL = False

OUT_ROOT = Path("tmp") / f"s_v2_pomdp_agent_training_allstates_fix3_e{EXPANSIONS}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ROOT


## 3. Load and inspect the environment

This mirrors the upstream training notebook pattern: first load/plot the environment, then create the agent.

`Environment.load(ENV_DIR)` is preferred when the environment folder already contains the metadata generated by olfactory-navigation. If that fails, the fallback constructs the environment from `data.npy` using the known source and margins.


In [ ]:
def load_training_environment(env_dir: Path):
    """Load env using upstream loader, with metadata-driven fallback.

    No source coordinates, margins or thresholds are hard-coded here.
    If Environment.load fails, the fallback reads the JSON metadata stored inside
    ENV_DIR and builds Environment(...) from those values.
    """
    return load_environment_from_metadata(
        env_dir,
        Environment,
        prefer_environment_load=True,
        verbose=True,
    )

# Optional audit: show which metadata file/kwargs would be used by the fallback.
try:
    env_meta = read_environment_metadata(ENV_DIR, verbose=True)
    env_kwargs_from_meta = environment_kwargs_from_metadata(ENV_DIR, env_meta, verbose=True)
except Exception as exc:
    print("Metadata audit skipped:", repr(exc))

env = load_training_environment(ENV_DIR)
normalize_environment_for_exact_converter(env, verbose=True)
print(env)
env.plot()

## 4. Agent factory

The upstream notebook uses `ag = FSVI_Agent(env)`. We keep the same semantics: no `partitions`, no `margin_partitions`, and no `minimal_converter`. This means the default exact converter uses all states of the environment.


In [ ]:
def make_agent(seed=SEED):
    print(f"[MAKE_AGENT] env_path={ENV_DIR}")
    print("[MAKE_AGENT] partitions=None / all environment states")
    print(f"[MAKE_AGENT] seed={seed}")
    normalize_environment_for_exact_converter(env, verbose=False)
    return FSVI_Agent(
        env,
        seed=int(seed),
    )

# Same role as the upstream notebook cell: ag = FSVI_Agent(env)
ag = make_agent()
print("[MAKE_AGENT] model_state_count=", getattr(ag.model, "state_count", None))
print("[MAKE_AGENT] action_count=", getattr(ag.model, "action_count", None))
print("[MAKE_AGENT] observation_count=", getattr(ag.model, "observation_count", None))
ag


## 5. S_v2 CUDA training with `traincuda`

This is the added path. It does not replace `ag.train(...)`; it wraps the normal agent and only replaces the expensive backup stage with S_v2 real-cuBLAS CUDA backup.


In [ ]:
if RUN_CUDA_TRAIN:
    if not CUDA_LIB.exists():
        raise FileNotFoundError(f"Missing CUDA_LIB={CUDA_LIB}. Build with: bash scripts/31_build_backend_lib.sh --arch 80 --clean")

    ag_cuda_base = make_agent()
    ag_cuda = enable_cuda_backend(
        ag_cuda_base,
        device=CUDA_DEVICE,
        version=CUDA_VERSION,
        gamma=GAMMA,
        lib_path=CUDA_LIB,
    )

    print("traincuda available:", hasattr(ag_cuda, "traincuda"))
    print("CUDA config:", ag_cuda._cuda_backend_config if hasattr(ag_cuda, "_cuda_backend_config") else {
        "lib_path": str(CUDA_LIB),
        "device": CUDA_DEVICE,
        "version": CUDA_VERSION,
        "gamma": GAMMA,
    })

    res_cuda = ag_cuda.traincuda(
        expansions=EXPANSIONS,
        use_gpu=True,
        gamma=GAMMA,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        outdir=str(OUT_ROOT / "cuda_traincuda"),
        checkpoint_every=max(1, EXPANSIONS // 4),
        visual=True,
        display_rows=12,
    )

    print(json.dumps(res_cuda.summary, indent=2))
    df_cuda = pd.DataFrame(res_cuda.rows)
else:
    ag_cuda = None
    res_cuda = None
    df_cuda = pd.DataFrame()
    print("CUDA training skipped")


## 6. Optional upstream CPU training

This remains the official `olfactory_navigation` path.


In [ ]:
if RUN_CPU_TRAIN:
    ag_cpu = make_agent()
    hist_cpu_train = ag_cpu.train(
        expansions=EXPANSIONS,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        gamma=GAMMA,
        use_gpu=False,
        overwrite_training=True,
        print_progress=True,
        print_stats=True,
    )
    print(hist_cpu_train.summary if hasattr(hist_cpu_train, "summary") else hist_cpu_train)
else:
    ag_cpu = None
    hist_cpu_train = None
    print("CPU/native training skipped")


## 7. Optional upstream CuPy training

Use this only when you want to compare with the original `use_gpu=True` path. On large environments this path can consume much more RAM/VRAM than the S_v2 backend.


In [ ]:
if RUN_CUPY_TRAIN:
    ag_cupy = make_agent()
    hist_cupy_train = ag_cupy.train(
        expansions=EXPANSIONS,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        gamma=GAMMA,
        use_gpu=True,
        overwrite_training=True,
        print_progress=True,
        print_stats=True,
    )
    print(hist_cupy_train.summary if hasattr(hist_cupy_train, "summary") else hist_cupy_train)
else:
    ag_cupy = None
    hist_cupy_train = None
    print("CuPy/native GPU training skipped")


## 8. Inspect S_v2 CUDA training rows

`actual_version` tells you whether S_v2 dispatched to `v7_cublas_all` or `v8_cublas_by_action`.


In [ ]:
if res_cuda is not None:
    show_cuda_training_report(res_cuda, display_rows=20)
    display_cols = [c for c in [
        "iter", "nB", "nG_in", "actual_version", "expand_ms", "backup_ms",
        "backup_wall_ms", "update_ms", "iter_total_ms", "alpha_after",
        "belief_total_after", "pruned"
    ] if c in df_cuda.columns]
    display(df_cuda[display_cols].tail(20))
else:
    print("No CUDA result available")


## 9. Generate policy start points

This cell creates shared start sets for CUDA, CPU and CuPy. It avoids hard-coded source coordinates and reads the starts from the loaded environment / agent metadata.

Generated objects:

- `start_points_raw`: raw coordinates from `env.start_probabilities > 0`
- `start_points_full`: all clean valid start points
- `start_points_eval`: subset controlled by `N_EVAL`


In [ ]:
# ============================================================
# START POINTS GENERATION
# ============================================================
# The reference agent can be CUDA, CPU, CuPy or the initial ag.
# The helper returns raw/full/eval arrays and saves them under OUT_ROOT.
# ============================================================

def _native_if_cuda(agent):
    if agent is None:
        return None
    return getattr(agent, "native_agent", agent)

# Pick the first available trained/reference agent.
ref_agent = None
for candidate_name in ["ag_cuda", "ag_cpu", "ag_cupy", "ag"]:
    candidate = globals().get(candidate_name, None)
    candidate = _native_if_cuda(candidate)
    if candidate is not None:
        ref_agent = candidate
        print(f"[START_POINTS] Using reference agent: {candidate_name}")
        break

if ref_agent is None:
    raise RuntimeError("No reference agent found. Create at least one agent first.")

start_point_sets = generate_policy_start_points(
    ref_agent,
    n_eval=N_EVAL,
    out_root=OUT_ROOT,
    remove_source_points=True,
    verbose=True,
)

start_points_raw = start_point_sets["raw"]
start_points_full = start_point_sets["full"]
start_points_eval = start_point_sets["eval"]

print("[START_POINTS] start_points_raw:", start_points_raw.shape)
print("[START_POINTS] start_points_full:", start_points_full.shape)
print("[START_POINTS] start_points_eval:", start_points_eval.shape)


## 10. Quick policy evaluation: CUDA / CPU / CuPy

Uses `start_points_eval`, so the three policies are tested on the same start set.

`use_gpu=False` is intentional here: the policy is already trained; this avoids mixing policy quality with simulator CPU/CuPy execution differences.

In [ ]:
# ============================================================
# QUICK POLICY EVALUATION: CUDA vs CPU vs CuPy
# ============================================================

def _print_hist_summary(label, hist):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    if hist is None:
        print("None")
    elif hasattr(hist, "summary"):
        print(hist.summary)
    else:
        print(hist)

COMMON_EVAL_KWARGS = dict(
    start_points=start_points_eval,
    n=len(start_points_eval),
    horizon=1000,
    reward_discount=GAMMA,
    use_gpu=False,
    time_shift=False,
    time_loop=False,
    print_progress=True,
    print_stats=True,
)

if RUN_POLICY_EVAL and res_cuda is not None:
    hist_cuda_eval = run_policy_evaluation(
        ag_cuda.native_agent,
        **COMMON_EVAL_KWARGS,
    )
    _print_hist_summary("CUDA policy evaluation", hist_cuda_eval)
else:
    hist_cuda_eval = None
    print("CUDA policy evaluation skipped")

if RUN_POLICY_EVAL and globals().get("hist_cpu_train", None) is not None and globals().get("ag_cpu", None) is not None:
    hist_cpu_eval = run_policy_evaluation(
        ag_cpu,
        **COMMON_EVAL_KWARGS,
    )
    _print_hist_summary("CPU/native policy evaluation", hist_cpu_eval)
else:
    hist_cpu_eval = None
    print("CPU/native policy evaluation skipped")

if RUN_POLICY_EVAL and globals().get("hist_cupy_train", None) is not None and globals().get("ag_cupy", None) is not None:
    hist_cupy_eval = run_policy_evaluation(
        ag_cupy,
        **COMMON_EVAL_KWARGS,
    )
    _print_hist_summary("CuPy/native GPU policy evaluation", hist_cupy_eval)
else:
    hist_cupy_eval = None
    print("CuPy/native GPU policy evaluation skipped")


## 11. Plot quick policy evaluation histories

In [ ]:
if hist_cuda_eval is not None:
    print("CUDA hist.plot()")
    hist_cuda_eval.plot()

if hist_cpu_eval is not None:
    print("CPU/native hist.plot()")
    hist_cpu_eval.plot()

if hist_cupy_eval is not None:
    print("CuPy/native GPU hist.plot()")
    hist_cupy_eval.plot()


## 12. Full policy evaluation: CUDA / CPU / CuPy

This uses `start_points_full`, i.e. all clean starts. It is disabled by default via `RUN_FULL_POLICY_EVAL=False` because the all-states environment can be expensive.

In [ ]:
# ============================================================
# FULL POLICY EVALUATION: CUDA vs CPU vs CuPy
# ============================================================

FULL_EVAL_KWARGS = dict(
    start_points=start_points_full,
    n=len(start_points_full),
    horizon=1000,
    reward_discount=GAMMA,
    use_gpu=False,
    time_shift=False,
    time_loop=False,
    print_progress=True,
    print_stats=True,
)

print("[FULL_EVAL] start_points_full:", start_points_full.shape)

if RUN_FULL_POLICY_EVAL and res_cuda is not None:
    hist_cuda_full = run_policy_evaluation(
        ag_cuda.native_agent,
        **FULL_EVAL_KWARGS,
    )
    _print_hist_summary("CUDA FULL policy evaluation", hist_cuda_full)
else:
    hist_cuda_full = None
    print("CUDA full policy evaluation skipped")

if RUN_FULL_POLICY_EVAL and globals().get("hist_cpu_train", None) is not None and globals().get("ag_cpu", None) is not None:
    hist_cpu_full = run_policy_evaluation(
        ag_cpu,
        **FULL_EVAL_KWARGS,
    )
    _print_hist_summary("CPU/native FULL policy evaluation", hist_cpu_full)
else:
    hist_cpu_full = None
    print("CPU/native full policy evaluation skipped")

if RUN_FULL_POLICY_EVAL and globals().get("hist_cupy_train", None) is not None and globals().get("ag_cupy", None) is not None:
    hist_cupy_full = run_policy_evaluation(
        ag_cupy,
        **FULL_EVAL_KWARGS,
    )
    _print_hist_summary("CuPy/native GPU FULL policy evaluation", hist_cupy_full)
else:
    hist_cupy_full = None
    print("CuPy/native GPU full policy evaluation skipped")


## 13. Plot full policy evaluation histories

In [ ]:
if hist_cuda_full is not None:
    print("CUDA FULL hist.plot()")
    hist_cuda_full.plot()

if hist_cpu_full is not None:
    print("CPU/native FULL hist.plot()")
    hist_cpu_full.plot()

if hist_cupy_full is not None:
    print("CuPy/native GPU FULL hist.plot()")
    hist_cupy_full.plot()


## 14. Save compact run metadata

In [ ]:
metadata = {
    "package_root": str(PACKAGE_ROOT),
    "cuda_lib": str(CUDA_LIB),
    "env_dir": str(ENV_DIR),
    "state_space": "all_environment_states",
    "spacial_subdivisions": None,
    "model_state_count": getattr(getattr(ag, "model", None), "state_count", None),
    "seed": SEED,
    "gamma": GAMMA,
    "expansions": EXPANSIONS,
    "cuda_version": CUDA_VERSION,
    "n_eval": None if N_EVAL is None else int(N_EVAL),
    "n_start_points_raw": int(len(start_points_raw)) if "start_points_raw" in globals() else None,
    "n_start_points_full": int(len(start_points_full)) if "start_points_full" in globals() else None,
    "n_start_points_eval": int(len(start_points_eval)) if "start_points_eval" in globals() else None,
    "run_full_policy_eval": bool(RUN_FULL_POLICY_EVAL),
    "cuda_summary": getattr(res_cuda, "summary", None),
}
(OUT_ROOT / "notebook_run_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True, default=str) + "\n")
print("Saved metadata:", OUT_ROOT / "notebook_run_metadata.json")
